# NSE 5× Turnaround — V4 Causal Portfolio Backtest
## Nifty Regime-Aware Strategy

This Colab notebook backtests the 5× Turnaround strategy with an explicit Nifty 50 market-regime indicator.

### Nifty source
`/content/drive/MyDrive/quant/data/indices/nifty50/NIFTY50.parquet`

### Causal execution
- Signal at close of T
- Entry at open of T+1
- Exit decision at close of T
- Exit execution at open of T+1
- No hard initial stop

### Nifty regimes
- **BULL**: Nifty close > 200DMA AND 50DMA > 200DMA AND 20D return > 0
- **BEAR**: Nifty close < 200DMA AND 50DMA < 200DMA AND 20D return < 0
- **MIXED**: everything else

The notebook evaluates no filter, Nifty > 200DMA, 50DMA > 200DMA, and full BULL regime filters across six trailing stops: 20%, 30%, 40%, 50%, 60%, 70%.

It also preserves the actual Nifty regime on every stock signal/trade for regime attribution.


In [ ]:
from pathlib import Path
import json
import warnings
from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

DATA_DIR = Path("/content/drive/MyDrive/quant/data/parquet")
NIFTY_PATH = Path("/content/drive/MyDrive/quant/data/indices/nifty50/NIFTY50.parquet")
RESULTS_DIR = Path("/content/drive/MyDrive/quant/data/results/5x_turnaround_v4")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_MIN_END_DATE = pd.Timestamp("2026-09-01")

STARTING_CAPITAL = 1_000_000.0
MAX_POSITIONS = 20
MAX_ALLOCATION_PER_POSITION = 1.0 / MAX_POSITIONS

SLIPPAGE_BPS = 5.0
BUY_COST_BPS = 5.0
SELL_COST_BPS = 15.0

ENTRY_SCORE_MIN = 8
TRAILING_STOPS = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]

FAILURE_EXIT_DAYS = 252
MAX_HOLD_DAYS = 756

WF_TRAIN_YEARS = 5
WF_VALID_YEARS = 2
WF_TEST_YEARS = 2

print("DATA_DIR:", DATA_DIR)
print("NIFTY_PATH:", NIFTY_PATH)
print("RESULTS_DIR:", RESULTS_DIR)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
print("Stock data exists:", DATA_DIR.exists(), DATA_DIR)
print("Nifty file exists:", NIFTY_PATH.exists(), NIFTY_PATH)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Stock data directory not found: {DATA_DIR}")

if not NIFTY_PATH.exists():
    raise FileNotFoundError(f"Nifty Parquet not found: {NIFTY_PATH}")

print("✓ Paths verified.")


In [ ]:
try:
    import pyarrow.parquet as pq
except ImportError:
    pq = None

def list_parquet_files(root: Path) -> List[Path]:
    return sorted(root.rglob("*.parquet"))

def normalize_ohlcv(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rename = {}
    for c in df.columns:
        lc = str(c).lower().strip()
        if lc in {"date", "timestamp", "datetime", "trade_date"}:
            rename[c] = "date"
        elif lc in {"symbol", "ticker", "tradingsymbol", "security"}:
            rename[c] = "symbol"
        elif lc in {"open", "open_price"}:
            rename[c] = "open"
        elif lc in {"high", "high_price"}:
            rename[c] = "high"
        elif lc in {"low", "low_price"}:
            rename[c] = "low"
        elif lc in {"close", "close_price", "last"}:
            rename[c] = "close"
        elif lc in {"volume", "vol"}:
            rename[c] = "volume"

    df = df.rename(columns=rename)

    required = {"date", "open", "high", "low", "close"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required OHLC columns: {sorted(missing)}")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    for c in ["open", "high", "low", "close", "volume"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=["date", "open", "high", "low", "close"])

    if "symbol" not in df.columns:
        df["symbol"] = "UNKNOWN"

    return (
        df.sort_values(["symbol", "date"])
          .drop_duplicates(["symbol", "date"], keep="last")
          .reset_index(drop=True)
    )

def parquet_date_bounds(path: Path):
    try:
        if pq is not None:
            pf = pq.ParquetFile(path)
            names = [x.lower() for x in pf.schema.names]
            date_col = None
            for candidate in ["date", "trade_date", "timestamp", "datetime"]:
                if candidate in names:
                    date_col = pf.schema.names[names.index(candidate)]
                    break
            if date_col:
                mins, maxs = [], []
                idx = names.index(date_col.lower())
                for rg in pf.metadata.row_groups:
                    st = rg.column(idx).statistics
                    if st and st.has_min_max:
                        mins.append(pd.to_datetime(st.min))
                        maxs.append(pd.to_datetime(st.max))
                if mins and maxs:
                    return min(mins), max(maxs)
    except Exception:
        pass

    try:
        df = pd.read_parquet(path, columns=["date"])
        s = pd.to_datetime(df["date"], errors="coerce").dropna()
        if len(s):
            return s.min(), s.max()
    except Exception:
        return None, None

    return None, None


In [ ]:
files = list_parquet_files(DATA_DIR)
print(f"Parquet files found: {len(files):,}")

if not files:
    raise FileNotFoundError(f"No Parquet files found below {DATA_DIR}")

bounds = []
for p in files:
    mn, mx = parquet_date_bounds(p)
    if mx is not None:
        bounds.append((p, mn, mx))

if not bounds:
    raise RuntimeError("Could not determine Parquet date bounds.")

actual_min = min(x[1] for x in bounds if x[1] is not None)
actual_max = max(x[2] for x in bounds)

print("Dataset minimum:", actual_min.date())
print("Dataset maximum:", actual_max.date())
print("Required minimum:", EXPECTED_MIN_END_DATE.date())

if actual_max < EXPECTED_MIN_END_DATE:
    raise RuntimeError(
        f"STALE DATASET FAILURE: latest stock date is {actual_max.date()}, "
        f"below required {EXPECTED_MIN_END_DATE.date()}. "
        "Run the incremental NSE downloader first."
    )

print("✓ Stock freshness guard passed.")


In [ ]:
frames = []

for i, p in enumerate(files, 1):
    try:
        x = normalize_ohlcv(pd.read_parquet(p))
        if len(x):
            frames.append(x)
    except Exception as e:
        print(f"Skipping {p.name}: {e}")

if not frames:
    raise RuntimeError("No readable Parquet data.")

stocks = normalize_ohlcv(pd.concat(frames, ignore_index=True))
stocks["symbol"] = stocks["symbol"].astype(str).str.upper().str.strip()
stocks = stocks.sort_values(["symbol", "date"]).reset_index(drop=True)

print(f"Rows: {len(stocks):,}")
print(f"Symbols: {stocks['symbol'].nunique():,}")
print(f"Range: {stocks['date'].min().date()} → {stocks['date'].max().date()}")

if stocks["date"].max() < EXPECTED_MIN_END_DATE:
    raise RuntimeError("Loaded stock dataset is stale.")

print("✓ Loaded stock dataset is fresh.")


In [ ]:
def load_nifty(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Nifty file does not exist: {path}")

    x = normalize_ohlcv(pd.read_parquet(path))

    x = (
        x[["date", "open", "high", "low", "close"]]
        .sort_values("date")
        .drop_duplicates("date", keep="last")
        .reset_index(drop=True)
    )
    return x

nifty = load_nifty(NIFTY_PATH)

print(f"Nifty rows: {len(nifty):,}")
print("Nifty range:", nifty["date"].min().date(), "→", nifty["date"].max().date())

if nifty["date"].max() < EXPECTED_MIN_END_DATE:
    raise RuntimeError(
        f"Nifty dataset is stale: {nifty['date'].max().date()} "
        f"< {EXPECTED_MIN_END_DATE.date()}"
    )

if nifty["date"].duplicated().any():
    raise ValueError("Duplicate Nifty dates found.")

if (nifty["close"] <= 0).any():
    raise ValueError("Invalid Nifty close values.")

print("✓ Nifty sanity checks passed.")


In [ ]:
# Nifty regime features
nifty["sma20"] = nifty["close"].rolling(20).mean()
nifty["sma50"] = nifty["close"].rolling(50).mean()
nifty["sma200"] = nifty["close"].rolling(200).mean()

nifty["ret20"] = nifty["close"].pct_change(20)
nifty["ret60"] = nifty["close"].pct_change(60)

nifty["nifty_above_200dma"] = nifty["close"] > nifty["sma200"]
nifty["nifty_50dma_above_200dma"] = nifty["sma50"] > nifty["sma200"]
nifty["nifty_positive_20d"] = nifty["ret20"] > 0

nifty["nifty_bull_regime"] = (
    nifty["nifty_above_200dma"]
    & nifty["nifty_50dma_above_200dma"]
    & nifty["nifty_positive_20d"]
)

nifty["nifty_bear_regime"] = (
    (nifty["close"] < nifty["sma200"])
    & (nifty["sma50"] < nifty["sma200"])
    & (nifty["ret20"] < 0)
)

nifty["nifty_mixed_regime"] = (
    ~nifty["nifty_bull_regime"]
    & ~nifty["nifty_bear_regime"]
)

nifty["nifty_regime"] = np.select(
    [
        nifty["nifty_bull_regime"],
        nifty["nifty_bear_regime"],
        nifty["nifty_mixed_regime"],
    ],
    ["BULL", "BEAR", "MIXED"],
    default="UNKNOWN"
)

display(nifty.tail(10))


In [ ]:
regime_distribution = (
    nifty["nifty_regime"]
    .value_counts(dropna=False)
    .rename_axis("regime")
    .reset_index(name="days")
)

regime_distribution["pct"] = (
    regime_distribution["days"] / regime_distribution["days"].sum()
)

display(regime_distribution)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 6))
plt.plot(nifty["date"], nifty["close"], label="Nifty Close")

bull = nifty[nifty["nifty_regime"] == "BULL"]
bear = nifty[nifty["nifty_regime"] == "BEAR"]

plt.scatter(bull["date"], bull["close"], s=3, label="BULL")
plt.scatter(bear["date"], bear["close"], s=3, label="BEAR")

plt.title("Nifty 50 Price with Market Regime")
plt.xlabel("Date")
plt.ylabel("Nifty Close")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


In [ ]:
NIFTY_REGIME_PATH = RESULTS_DIR / "nifty50_regime.parquet"
nifty.to_parquet(NIFTY_REGIME_PATH, index=False)
print("Saved:", NIFTY_REGIME_PATH)


In [ ]:
LOOKBACK_60 = 60
LOOKBACK_252 = 252

def add_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("date").copy()

    g["sma20"] = g["close"].rolling(20).mean()
    g["sma50"] = g["close"].rolling(50).mean()
    g["sma100"] = g["close"].rolling(100).mean()
    g["sma200"] = g["close"].rolling(200).mean()

    g["high60_prev"] = g["high"].rolling(LOOKBACK_60).max().shift(1)
    g["high252_prev"] = g["high"].rolling(LOOKBACK_252).max().shift(1)
    g["low252_prev"] = g["low"].rolling(LOOKBACK_252).min().shift(1)

    g["ret20"] = g["close"].pct_change(20)
    g["ret60"] = g["close"].pct_change(60)
    g["ret120"] = g["close"].pct_change(120)
    g["ret252"] = g["close"].pct_change(252)

    g["vol60"] = g["close"].pct_change().rolling(60).std()

    if "volume" in g.columns:
        g["vol_avg60"] = g["volume"].rolling(60).mean()
        g["volume_ratio60"] = g["volume"] / g["vol_avg60"]
    else:
        g["volume_ratio60"] = np.nan

    g["drawdown252"] = g["close"] / g["high252_prev"] - 1.0
    g["range252"] = g["high252_prev"] / g["low252_prev"] - 1.0

    g["above20"] = g["close"] > g["sma20"]
    g["sma20_rising"] = g["sma20"] > g["sma20"].shift(5)
    g["higher_low"] = g["low"] > g["low"].shift(5)
    g["breakout60"] = g["close"] > g["high60_prev"]
    g["above50"] = g["close"] > g["sma50"]
    g["sma50_gt100"] = g["sma50"] > g["sma100"]
    g["positive_momentum"] = g["ret20"] > 0
    g["volume_expansion"] = g["volume_ratio60"] > 1.5
    g["above200"] = g["close"] > g["sma200"]

    g["score"] = (
        g["above20"].fillna(False).astype(int)
        + g["sma20_rising"].fillna(False).astype(int)
        + g["higher_low"].fillna(False).astype(int)
        + g["breakout60"].fillna(False).astype(int)
        + g["above50"].fillna(False).astype(int)
        + g["sma50_gt100"].fillna(False).astype(int)
        + g["positive_momentum"].fillna(False).astype(int)
        + g["volume_expansion"].fillna(False).astype(int)
        + g["above200"].fillna(False).astype(int)
    )

    g["entry_signal"] = (
        (g["score"] >= ENTRY_SCORE_MIN)
        & g["breakout60"]
        & g["above50"]
        & g["positive_momentum"]
        & g["sma50_gt100"]
    )

    return g

features = (
    stocks.groupby("symbol", group_keys=False)
          .apply(add_features, include_groups=False)
          .reset_index(drop=True)
)

print("Feature rows:", f"{len(features):,}")
print("Raw entry signals:", f"{int(features['entry_signal'].sum()):,}")


In [ ]:
nifty_features = nifty[
    [
        "date",
        "close",
        "sma50",
        "sma200",
        "ret20",
        "nifty_above_200dma",
        "nifty_50dma_above_200dma",
        "nifty_positive_20d",
        "nifty_bull_regime",
        "nifty_bear_regime",
        "nifty_mixed_regime",
        "nifty_regime",
    ]
].rename(
    columns={
        "close": "nifty_close",
        "sma50": "nifty_sma50",
        "sma200": "nifty_sma200",
        "ret20": "nifty_ret20",
    }
)

features = features.merge(
    nifty_features,
    on="date",
    how="left",
    validate="many_to_one"
)

features["nifty_regime"] = features["nifty_regime"].fillna("UNKNOWN")
features["nifty_bull_regime"] = features["nifty_bull_regime"].fillna(False)
features["nifty_bear_regime"] = features["nifty_bear_regime"].fillna(False)
features["nifty_mixed_regime"] = features["nifty_mixed_regime"].fillna(False)

print("Merged rows:", f"{len(features):,}")
print(features["nifty_regime"].value_counts(dropna=False))


In [ ]:
REGIMES = ["none", "nifty_gt_200", "50_gt_200", "bull"]

def regime_filter(df: pd.DataFrame, regime: str) -> pd.Series:
    if regime == "none":
        return pd.Series(True, index=df.index)
    if regime == "nifty_gt_200":
        return df["nifty_above_200dma"].fillna(False)
    if regime == "50_gt_200":
        return df["nifty_50dma_above_200dma"].fillna(False)
    if regime == "bull":
        return df["nifty_bull_regime"].fillna(False)
    raise ValueError(f"Unknown regime: {regime}")

signal_regime = (
    features[features["entry_signal"]]
    .groupby("nifty_regime")
    .size()
    .reset_index(name="signals")
)

signal_regime["pct"] = signal_regime["signals"] / signal_regime["signals"].sum()
display(signal_regime)


In [ ]:
@dataclass
class Trade:
    symbol: str
    signal_date: pd.Timestamp
    entry_date: pd.Timestamp
    entry_price: float
    exit_date: pd.Timestamp
    exit_price: float
    exit_reason: str
    gross_return: float
    net_return: float
    mae: float
    mfe: float
    hold_days: int
    score: float
    nifty_regime: str
    nifty_close: float
    nifty_sma50: float
    nifty_sma200: float
    nifty_ret20: float

def apply_slippage(price: float, side: str) -> float:
    if side == "buy":
        return price * (1.0 + SLIPPAGE_BPS / 10000.0)
    return price * (1.0 - SLIPPAGE_BPS / 10000.0)

def net_trade_return(entry_raw: float, exit_raw: float) -> float:
    entry = apply_slippage(entry_raw, "buy") * (1.0 + BUY_COST_BPS / 10000.0)
    exit_ = apply_slippage(exit_raw, "sell") * (1.0 - SELL_COST_BPS / 10000.0)
    return exit_ / entry - 1.0


In [ ]:
def simulate_trade(g: pd.DataFrame, signal_idx: int, trailing_stop: float) -> Optional[Trade]:
    if signal_idx + 1 >= len(g):
        return None

    entry_idx = signal_idx + 1
    entry_row = g.iloc[entry_idx]
    entry_raw = float(entry_row["open"])

    if not np.isfinite(entry_raw) or entry_raw <= 0:
        return None

    entry_price = apply_slippage(entry_raw, "buy")
    mae = 0.0
    mfe = 0.0
    last_idx = min(len(g) - 1, entry_idx + MAX_HOLD_DAYS)

    for i in range(entry_idx, last_idx + 1):
        row = g.iloc[i]
        hi = float(row["high"])
        lo = float(row["low"])

        mae = min(mae, lo / entry_raw - 1.0)
        mfe = max(mfe, hi / entry_raw - 1.0)

        if i >= last_idx:
            break

        current_close = float(row["close"])
        current_sma50 = float(row["sma50"]) if pd.notna(row["sma50"]) else np.nan

        highest_close = float(g.iloc[entry_idx:i + 1]["close"].max())
        stop_price = highest_close * (1.0 - trailing_stop)

        reason = None

        if current_close <= stop_price:
            reason = "TRAILING_STOP"

        elif (
            (i - entry_idx + 1) >= FAILURE_EXIT_DAYS
            and np.isfinite(current_sma50)
            and current_close < current_sma50
            and current_close < entry_price
        ):
            reason = "FAILURE_EXIT"

        elif (i - entry_idx + 1) >= MAX_HOLD_DAYS:
            reason = "TIME_EXIT"

        if reason:
            exit_idx = i + 1
            exit_row = g.iloc[exit_idx]
            exit_raw = float(exit_row["open"])

            if not np.isfinite(exit_raw) or exit_raw <= 0:
                continue

            gross = exit_raw / entry_raw - 1.0
            net = net_trade_return(entry_raw, exit_raw)
            signal_row = g.iloc[signal_idx]

            return Trade(
                symbol=str(signal_row["symbol"]),
                signal_date=pd.Timestamp(signal_row["date"]),
                entry_date=pd.Timestamp(entry_row["date"]),
                entry_price=float(entry_price),
                exit_date=pd.Timestamp(exit_row["date"]),
                exit_price=float(apply_slippage(exit_raw, "sell")),
                exit_reason=reason,
                gross_return=float(gross),
                net_return=float(net),
                mae=float(mae),
                mfe=float(mfe),
                hold_days=int((exit_row["date"] - entry_row["date"]).days),
                score=float(signal_row["score"]),
                nifty_regime=str(signal_row["nifty_regime"]),
                nifty_close=float(signal_row["nifty_close"]),
                nifty_sma50=float(signal_row["nifty_sma50"]),
                nifty_sma200=float(signal_row["nifty_sma200"]),
                nifty_ret20=float(signal_row["nifty_ret20"]),
            )

    return None


In [ ]:
def build_trade_set(features: pd.DataFrame, trailing_stop: float, regime: str) -> pd.DataFrame:
    if regime not in REGIMES:
        raise ValueError(regime)

    records = []

    for symbol, g in features.groupby("symbol", sort=False):
        g = g.sort_values("date").reset_index(drop=True)
        eligible = g["entry_signal"] & regime_filter(g, regime)

        for idx in np.flatnonzero(eligible.to_numpy()):
            trade = simulate_trade(g, int(idx), trailing_stop)
            if trade is not None:
                records.append(trade.__dict__)

    cols = [
        "symbol", "signal_date", "entry_date", "entry_price",
        "exit_date", "exit_price", "exit_reason",
        "gross_return", "net_return", "mae", "mfe", "hold_days", "score",
        "nifty_regime", "nifty_close", "nifty_sma50",
        "nifty_sma200", "nifty_ret20"
    ]

    if not records:
        return pd.DataFrame(columns=cols)

    return (
        pd.DataFrame(records)
        .sort_values(["entry_date", "symbol"])
        .reset_index(drop=True)
    )

trade_sets = {}

for regime in REGIMES:
    for stop in TRAILING_STOPS:
        print(f"Building regime={regime}, stop={stop:.0%}")
        trade_sets[(regime, stop)] = build_trade_set(features, stop, regime)
        print("  causal closed trades:", len(trade_sets[(regime, stop)]))


In [ ]:
trade_count_matrix = pd.DataFrame([
    {
        "regime": regime,
        "trailing_stop": stop,
        "trades": len(trade_sets[(regime, stop)])
    }
    for regime in REGIMES
    for stop in TRAILING_STOPS
])

display(trade_count_matrix)


In [ ]:
def simulate_portfolio(features: pd.DataFrame, trades: pd.DataFrame):
    if trades.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    entry_map = {}
    exit_map = {}

    for idx, r in trades.iterrows():
        entry_map.setdefault(pd.Timestamp(r["entry_date"]), []).append((idx, r))
        exit_map.setdefault(pd.Timestamp(r["exit_date"]), []).append((idx, r))

    px = (
        features.set_index(["date", "symbol"])[["open", "close"]]
        .sort_index()
    )

    dates = sorted(pd.to_datetime(features["date"].unique()))
    cash = STARTING_CAPITAL
    positions = {}
    closed = []
    equity_rows = []

    for date in dates:
        date = pd.Timestamp(date)

        # Exit first
        for trade_id, r in exit_map.get(date, []):
            symbol = str(r["symbol"])
            if symbol not in positions:
                continue

            p = positions.pop(symbol)
            qty = p["qty"]
            exit_px = float(r["exit_price"])
            gross = qty * exit_px
            sell_cost = gross * SELL_COST_BPS / 10000.0
            proceeds = gross - sell_cost
            cash += proceeds

            closed.append({
                **r.to_dict(),
                "portfolio_entry_price": p["entry_price"],
                "portfolio_exit_price": exit_px,
                "quantity": qty,
                "notional_entry": p["notional"],
                "notional_exit": gross,
                "portfolio_net_return": proceeds / p["cash_deployed"] - 1.0,
            })

        # Enter second
        for trade_id, r in entry_map.get(date, []):
            symbol = str(r["symbol"])

            if symbol in positions or len(positions) >= MAX_POSITIONS:
                continue

            try:
                open_raw = float(px.loc[(date, symbol), "open"])
            except Exception:
                continue

            if not np.isfinite(open_raw) or open_raw <= 0:
                continue

            current_equity = cash + sum(
                p["qty"] * p["last_close"] for p in positions.values()
            )

            target = min(
                current_equity * MAX_ALLOCATION_PER_POSITION,
                cash / (1.0 + BUY_COST_BPS / 10000.0)
            )

            if target <= 0:
                continue

            buy_px = apply_slippage(open_raw, "buy")
            buy_cost = target * BUY_COST_BPS / 10000.0
            qty = target / buy_px
            cash -= target + buy_cost

            positions[symbol] = {
                "entry_price": buy_px,
                "qty": qty,
                "notional": target,
                "cash_deployed": target + buy_cost,
                "last_close": buy_px,
            }

        # Mark
        for symbol, p in positions.items():
            try:
                close_px = float(px.loc[(date, symbol), "close"])
                if np.isfinite(close_px):
                    p["last_close"] = close_px
            except Exception:
                pass

        position_value = sum(
            p["qty"] * p["last_close"] for p in positions.values()
        )

        equity = cash + position_value

        equity_rows.append({
            "date": date,
            "equity": equity,
            "cash": cash,
            "position_value": position_value,
            "open_positions": len(positions),
            "exposure": position_value / equity if equity > 0 else 0.0,
        })

    open_rows = [
        {
            "symbol": symbol,
            "entry_price": p["entry_price"],
            "quantity": p["qty"],
            "last_close": p["last_close"],
            "unrealized_return": p["last_close"] / p["entry_price"] - 1.0,
        }
        for symbol, p in positions.items()
    ]

    return pd.DataFrame(equity_rows), pd.DataFrame(closed), pd.DataFrame(open_rows)


In [ ]:
def max_drawdown(equity):
    peak = equity.cummax()
    return float((equity / peak - 1.0).min())

def annualized_sharpe(equity):
    r = equity.pct_change().dropna()
    if len(r) < 2 or r.std() == 0:
        return np.nan
    return float(np.sqrt(252) * r.mean() / r.std())

def annualized_sortino(equity):
    r = equity.pct_change().dropna()
    downside = r[r < 0]
    if len(downside) < 2 or downside.std() == 0:
        return np.nan
    return float(np.sqrt(252) * r.mean() / downside.std())

def portfolio_metrics(equity_df, closed_df):
    if equity_df.empty:
        return {}

    e = equity_df.sort_values("date").copy()
    end = float(e["equity"].iloc[-1])
    days = max((e["date"].iloc[-1] - e["date"].iloc[0]).days, 1)
    years = days / 365.25
    cagr = (end / STARTING_CAPITAL) ** (1.0 / years) - 1.0
    dd = max_drawdown(e["equity"])

    return {
        "start_date": e["date"].iloc[0],
        "end_date": e["date"].iloc[-1],
        "starting_capital": STARTING_CAPITAL,
        "ending_equity": end,
        "CAGR": cagr,
        "max_drawdown": dd,
        "Sharpe": annualized_sharpe(e["equity"]),
        "Sortino": annualized_sortino(e["equity"]),
        "Calmar": cagr / abs(dd) if dd < 0 else np.nan,
        "closed_trades": int(len(closed_df)),
        "avg_open_positions": float(e["open_positions"].mean()),
        "max_open_positions": int(e["open_positions"].max()),
        "avg_exposure": float(e["exposure"].mean()),
        "max_exposure": float(e["exposure"].max()),
    }

def calendar_year_returns(equity_df):
    e = equity_df.sort_values("date").copy()
    e["year"] = e["date"].dt.year

    year_end = (
        e.groupby("year", as_index=False)
         .tail(1)[["year", "date", "equity"]]
         .sort_values("year")
         .reset_index(drop=True)
    )

    rows = []
    previous = STARTING_CAPITAL

    for _, r in year_end.iterrows():
        rows.append({
            "year": int(r["year"]),
            "year_end_date": r["date"],
            "year_end_equity": r["equity"],
            "return": r["equity"] / previous - 1.0,
        })
        previous = r["equity"]

    return pd.DataFrame(rows)


In [ ]:
matrix_rows = {}
portfolio_store = {}

for (regime, stop), trades in trade_sets.items():
    print(f"Backtesting regime={regime}, trailing_stop={stop:.0%}")

    eq, closed, open_pos = simulate_portfolio(features, trades)
    if eq.empty:
        continue

    metrics = portfolio_metrics(eq, closed)

    matrix_rows[(regime, stop)] = {
        "regime": regime,
        "trailing_stop": stop,
        **metrics,
    }

    portfolio_store[(regime, stop)] = {
        "equity": eq,
        "closed": closed,
        "open": open_pos,
        "yearly": calendar_year_returns(eq),
    }

matrix = pd.DataFrame(matrix_rows.values())

display_cols = [
    "regime", "trailing_stop", "ending_equity", "CAGR",
    "max_drawdown", "Sharpe", "Sortino", "Calmar",
    "closed_trades", "avg_open_positions",
    "max_open_positions", "avg_exposure", "max_exposure"
]

display(
    matrix[display_cols]
    .sort_values(["regime", "trailing_stop"])
    .reset_index(drop=True)
)


In [ ]:
print("CAGR matrix")
display(
    matrix.pivot(index="regime", columns="trailing_stop", values="CAGR")
    .style.format("{:.2%}")
)

print("Max drawdown matrix")
display(
    matrix.pivot(index="regime", columns="trailing_stop", values="max_drawdown")
    .style.format("{:.2%}")
)

print("Sharpe matrix")
display(
    matrix.pivot(index="regime", columns="trailing_stop", values="Sharpe")
    .style.format("{:.2f}")
)


In [ ]:
matrix.to_csv(
    RESULTS_DIR / "v4_nifty_regime_trailing_stop_matrix.csv",
    index=False
)

for (regime, stop), bundle in portfolio_store.items():
    tag = regime.replace(">", "gt").replace(" ", "_")
    stop_tag = f"{int(stop * 100):02d}"
    prefix = f"{tag}_stop_{stop_tag}"

    bundle["equity"].to_parquet(
        RESULTS_DIR / f"{prefix}_equity.parquet",
        index=False
    )
    bundle["closed"].to_parquet(
        RESULTS_DIR / f"{prefix}_closed_trades.parquet",
        index=False
    )
    bundle["open"].to_parquet(
        RESULTS_DIR / f"{prefix}_open_positions.parquet",
        index=False
    )
    bundle["yearly"].to_csv(
        RESULTS_DIR / f"{prefix}_yearly.csv",
        index=False
    )

print("Exported results to:", RESULTS_DIR)


In [ ]:
def trade_diagnostics(trades):
    if trades.empty:
        return {}

    return {
        "trades": len(trades),
        "win_rate": float((trades["net_return"] > 0).mean()),
        "mean_return": float(trades["net_return"].mean()),
        "median_return": float(trades["net_return"].median()),
        "avg_mae": float(trades["mae"].mean()),
        "avg_mfe": float(trades["mfe"].mean()),
        "2x_rate": float((trades["mfe"] >= 1.0).mean()),
        "3x_rate": float((trades["mfe"] >= 2.0).mean()),
        "5x_rate": float((trades["mfe"] >= 4.0).mean()),
        "median_hold_days": float(trades["hold_days"].median()),
    }

diag_rows = []

for (regime, stop), trades in trade_sets.items():
    d = trade_diagnostics(trades)
    if d:
        diag_rows.append({
            "regime": regime,
            "trailing_stop": stop,
            **d
        })

diagnostics = pd.DataFrame(diag_rows)
display(diagnostics.sort_values(["regime", "trailing_stop"]).reset_index(drop=True))

diagnostics.to_csv(
    RESULTS_DIR / "v4_trade_diagnostics.csv",
    index=False
)


In [ ]:
# Actual market-regime attribution using the unfiltered 40% strategy
base_trades = trade_sets[("none", 0.40)].copy()

if not base_trades.empty:
    regime_trade_stats = (
        base_trades.groupby("nifty_regime")
        .agg(
            trades=("symbol", "count"),
            win_rate=("net_return", lambda x: (x > 0).mean()),
            mean_return=("net_return", "mean"),
            median_return=("net_return", "median"),
            avg_mae=("mae", "mean"),
            avg_mfe=("mfe", "mean"),
            hit_2x=("mfe", lambda x: (x >= 1).mean()),
            hit_3x=("mfe", lambda x: (x >= 2).mean()),
            hit_5x=("mfe", lambda x: (x >= 4).mean()),
            median_hold_days=("hold_days", "median"),
        )
        .reset_index()
    )

    display(regime_trade_stats)

    regime_trade_stats.to_csv(
        RESULTS_DIR / "v4_trade_performance_by_actual_nifty_regime.csv",
        index=False
    )


In [ ]:
# Calendar-year returns for every configuration
yearly_rows = []

for (regime, stop), bundle in portfolio_store.items():
    yearly = bundle["yearly"].copy()
    yearly["regime"] = regime
    yearly["trailing_stop"] = stop
    yearly_rows.append(yearly)

if yearly_rows:
    all_yearly = pd.concat(yearly_rows, ignore_index=True)

    display(
        all_yearly[
            ["year", "regime", "trailing_stop", "return", "year_end_equity"]
        ].sort_values(["year", "regime", "trailing_stop"])
    )

    all_yearly.to_csv(
        RESULTS_DIR / "v4_all_calendar_year_returns.csv",
        index=False
    )


In [ ]:
def make_walk_forward_windows(start, end):
    windows = []
    cursor = pd.Timestamp(start).normalize()

    while True:
        train_end = (
            cursor
            + pd.DateOffset(years=WF_TRAIN_YEARS)
            - pd.Timedelta(days=1)
        )
        valid_end = train_end + pd.DateOffset(years=WF_VALID_YEARS)
        test_end = valid_end + pd.DateOffset(years=WF_TEST_YEARS)

        if test_end > end:
            break

        windows.append({
            "train_start": cursor,
            "train_end": train_end,
            "valid_start": train_end + pd.Timedelta(days=1),
            "valid_end": valid_end,
            "test_start": valid_end + pd.Timedelta(days=1),
            "test_end": test_end,
        })

        cursor = cursor + pd.DateOffset(years=WF_TEST_YEARS)

    return windows

wf_windows = make_walk_forward_windows(
    features["date"].min(),
    features["date"].max()
)

print("Walk-forward folds:", len(wf_windows))

for i, w in enumerate(wf_windows, 1):
    print(
        i,
        "| train", w["train_start"].date(), "→", w["train_end"].date(),
        "| valid", w["valid_start"].date(), "→", w["valid_end"].date(),
        "| test", w["test_start"].date(), "→", w["test_end"].date(),
    )


In [ ]:
def window_trade_stats(trades, start, end):
    x = trades[
        (pd.to_datetime(trades["entry_date"]) >= start)
        & (pd.to_datetime(trades["entry_date"]) <= end)
        & (pd.to_datetime(trades["exit_date"]) <= end)
    ].copy()

    if x.empty:
        return {
            "trades": 0,
            "win_rate": np.nan,
            "mean_return": np.nan,
            "median_return": np.nan,
            "2x_rate": np.nan,
            "3x_rate": np.nan,
            "5x_rate": np.nan,
        }

    return {
        "trades": len(x),
        "win_rate": float((x["net_return"] > 0).mean()),
        "mean_return": float(x["net_return"].mean()),
        "median_return": float(x["net_return"].median()),
        "2x_rate": float((x["mfe"] >= 1.0).mean()),
        "3x_rate": float((x["mfe"] >= 2.0).mean()),
        "5x_rate": float((x["mfe"] >= 4.0).mean()),
    }

wf_rows = []

for fold, w in enumerate(wf_windows, 1):
    for regime in REGIMES:
        for stop in TRAILING_STOPS:
            trades = trade_sets[(regime, stop)]

            train = window_trade_stats(trades, w["train_start"], w["train_end"])
            valid = window_trade_stats(trades, w["valid_start"], w["valid_end"])
            test = window_trade_stats(trades, w["test_start"], w["test_end"])

            wf_rows.append({
                "fold": fold,
                "regime": regime,
                "trailing_stop": stop,
                "train_start": w["train_start"],
                "train_end": w["train_end"],
                "valid_start": w["valid_start"],
                "valid_end": w["valid_end"],
                "test_start": w["test_start"],
                "test_end": w["test_end"],
                **{f"train_{k}": v for k, v in train.items()},
                **{f"valid_{k}": v for k, v in valid.items()},
                **{f"test_{k}": v for k, v in test.items()},
            })

wf = pd.DataFrame(wf_rows)

display(
    wf[
        [
            "fold", "regime", "trailing_stop",
            "test_start", "test_end",
            "test_trades", "test_win_rate",
            "test_mean_return", "test_median_return",
            "test_2x_rate", "test_3x_rate", "test_5x_rate"
        ]
    ].sort_values(["fold", "regime", "trailing_stop"])
)

wf.to_csv(
    RESULTS_DIR / "v4_walk_forward_trade_validation.csv",
    index=False
)


In [ ]:
# Walk-forward portfolio test
def simulate_portfolio_subset(features, trades, start, end):
    x = trades[
        (pd.to_datetime(trades["entry_date"]) >= start)
        & (pd.to_datetime(trades["entry_date"]) <= end)
        & (pd.to_datetime(trades["exit_date"]) <= end)
    ].copy()

    if x.empty:
        return pd.DataFrame(), pd.DataFrame()

    entry_map = {}
    exit_map = {}

    for idx, r in x.iterrows():
        entry_map.setdefault(pd.Timestamp(r["entry_date"]), []).append((idx, r))
        exit_map.setdefault(pd.Timestamp(r["exit_date"]), []).append((idx, r))

    px = features.set_index(["date", "symbol"])[["open", "close"]].sort_index()

    all_dates = pd.to_datetime(features["date"].unique())
    dates = sorted(d for d in all_dates if start <= d <= end)

    cash = STARTING_CAPITAL
    positions = {}
    closed = []
    eq_rows = []

    for date in dates:
        date = pd.Timestamp(date)

        for trade_id, r in exit_map.get(date, []):
            symbol = str(r["symbol"])
            if symbol not in positions:
                continue

            p = positions.pop(symbol)
            gross = p["qty"] * float(r["exit_price"])
            sell_cost = gross * SELL_COST_BPS / 10000.0
            cash += gross - sell_cost
            closed.append(r.to_dict())

        for trade_id, r in entry_map.get(date, []):
            symbol = str(r["symbol"])

            if symbol in positions or len(positions) >= MAX_POSITIONS:
                continue

            try:
                open_raw = float(px.loc[(date, symbol), "open"])
            except Exception:
                continue

            if not np.isfinite(open_raw) or open_raw <= 0:
                continue

            current_equity = cash + sum(
                p["qty"] * p["last_close"] for p in positions.values()
            )

            target = min(
                current_equity * MAX_ALLOCATION_PER_POSITION,
                cash / (1 + BUY_COST_BPS / 10000.0)
            )

            if target <= 0:
                continue

            buy_px = apply_slippage(open_raw, "buy")
            cost = target * BUY_COST_BPS / 10000.0
            qty = target / buy_px
            cash -= target + cost

            positions[symbol] = {
                "qty": qty,
                "entry_price": buy_px,
                "last_close": buy_px,
            }

        for symbol, p in positions.items():
            try:
                close_px = float(px.loc[(date, symbol), "close"])
                if np.isfinite(close_px):
                    p["last_close"] = close_px
            except Exception:
                pass

        position_value = sum(
            p["qty"] * p["last_close"] for p in positions.values()
        )

        eq_rows.append({
            "date": date,
            "equity": cash + position_value,
            "open_positions": len(positions),
        })

    return pd.DataFrame(eq_rows), pd.DataFrame(closed)

wf_port_rows = []

for fold, w in enumerate(wf_windows, 1):
    for regime in REGIMES:
        for stop in TRAILING_STOPS:
            trades = trade_sets[(regime, stop)]

            eq, closed = simulate_portfolio_subset(
                features, trades, w["test_start"], w["test_end"]
            )

            if eq.empty:
                continue

            m = portfolio_metrics(eq, closed)

            wf_port_rows.append({
                "fold": fold,
                "regime": regime,
                "trailing_stop": stop,
                **m,
            })

wf_portfolio = pd.DataFrame(wf_port_rows)

if not wf_portfolio.empty:
    display(
        wf_portfolio[
            [
                "fold", "regime", "trailing_stop",
                "ending_equity", "CAGR", "max_drawdown",
                "Sharpe", "Sortino", "Calmar",
                "closed_trades", "avg_open_positions"
            ]
        ].sort_values(["fold", "regime", "trailing_stop"])
    )

wf_portfolio.to_csv(
    RESULTS_DIR / "v4_walk_forward_portfolio_validation.csv",
    index=False
)


In [ ]:
if not wf_portfolio.empty:
    wf_summary = (
        wf_portfolio
        .groupby(["regime", "trailing_stop"], as_index=False)
        .agg(
            folds=("fold", "count"),
            mean_CAGR=("CAGR", "mean"),
            median_CAGR=("CAGR", "median"),
            mean_max_drawdown=("max_drawdown", "mean"),
            worst_max_drawdown=("max_drawdown", "min"),
            mean_Sharpe=("Sharpe", "mean"),
            median_Sharpe=("Sharpe", "median"),
            mean_Sortino=("Sortino", "mean"),
            mean_Calmar=("Calmar", "mean"),
            total_closed_trades=("closed_trades", "sum"),
        )
    )

    display(wf_summary.sort_values(["regime", "trailing_stop"]))

    wf_summary.to_csv(
        RESULTS_DIR / "v4_walk_forward_summary.csv",
        index=False
    )
else:
    wf_summary = pd.DataFrame()
    print("No walk-forward portfolio results.")


In [ ]:
# Equity curve comparison — 40% trailing stop
plt.figure(figsize=(16, 7))

for regime in REGIMES:
    key = (regime, 0.40)

    if key not in portfolio_store:
        continue

    eq = portfolio_store[key]["equity"]
    normalized = eq["equity"] / STARTING_CAPITAL

    plt.plot(eq["date"], normalized, label=regime)

plt.title("5× Strategy — 40% Trailing Stop by Nifty Filter")
plt.xlabel("Date")
plt.ylabel("Portfolio Value / Starting Capital")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


In [ ]:
# Nifty regime visualization
plt.figure(figsize=(16, 7))

for regime in ["BULL", "MIXED", "BEAR"]:
    x = nifty[nifty["nifty_regime"] == regime]
    plt.scatter(x["date"], x["close"], s=4, label=regime)

plt.plot(nifty["date"], nifty["sma200"], label="200 DMA")
plt.title("Nifty Regime Classification")
plt.xlabel("Date")
plt.ylabel("Nifty")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


In [ ]:
# Causal implementation checks
all_trade_sets = [x for x in trade_sets.values() if not x.empty]

if all_trade_sets:
    all_trades = pd.concat(all_trade_sets, ignore_index=True)

    assert (
        pd.to_datetime(all_trades["entry_date"])
        > pd.to_datetime(all_trades["signal_date"])
    ).all()

    assert (
        pd.to_datetime(all_trades["exit_date"])
        > pd.to_datetime(all_trades["entry_date"])
    ).all()

    assert (all_trades["entry_price"] > 0).all()
    assert (all_trades["exit_price"] > 0).all()
    assert (all_trades["hold_days"] >= 1).all()

    print("✓ signal_date < entry_date")
    print("✓ entry_date < exit_date")
    print("✓ positive prices")
    print("✓ positive holding periods")
    print("✓ Nifty regime attached to trades")

print("✓ Stock freshness:", actual_max.date(), ">=", EXPECTED_MIN_END_DATE.date())
print("✓ Nifty freshness:", nifty["date"].max().date(), ">=", EXPECTED_MIN_END_DATE.date())


In [ ]:
# DuckDB sanity checks
try:
    import duckdb

    con = duckdb.connect()

    display(
        con.execute(
            f'''
            SELECT
                COUNT(*) AS rows,
                MIN(date) AS min_date,
                MAX(date) AS max_date
            FROM read_parquet('{NIFTY_REGIME_PATH}')
            '''
        ).df()
    )

    display(
        con.execute(
            f'''
            SELECT COUNT(*) AS duplicate_dates
            FROM (
                SELECT date
                FROM read_parquet('{NIFTY_REGIME_PATH}')
                GROUP BY date
                HAVING COUNT(*) > 1
            )
            '''
        ).df()
    )

    display(
        con.execute(
            f'''
            SELECT nifty_regime, COUNT(*) AS days
            FROM read_parquet('{NIFTY_REGIME_PATH}')
            GROUP BY nifty_regime
            ORDER BY days DESC
            '''
        ).df()
    )

    con.close()
    print("✓ DuckDB Nifty sanity checks passed.")

except ImportError:
    print("DuckDB not installed. Run: !pip install duckdb")


In [ ]:
# Save complete feature dataset
FEATURES_PATH = RESULTS_DIR / "stock_features_with_nifty_regime.parquet"
features.to_parquet(FEATURES_PATH, index=False)
print("Saved:", FEATURES_PATH)


In [ ]:
# Save all trade sets
TRADES_DIR = RESULTS_DIR / "trades"
TRADES_DIR.mkdir(parents=True, exist_ok=True)

for (regime, stop), trades in trade_sets.items():
    tag = regime.replace(">", "gt").replace(" ", "_")
    stop_tag = f"{int(stop * 100):02d}"
    path = TRADES_DIR / f"{tag}_stop_{stop_tag}_trades.parquet"
    trades.to_parquet(path, index=False)

print("Trade sets saved to:", TRADES_DIR)


In [ ]:
manifest = {
    "version": "V4_NIFTY_REGIME",
    "data_dir": str(DATA_DIR),
    "nifty_path": str(NIFTY_PATH),
    "results_dir": str(RESULTS_DIR),
    "actual_stock_min_date": str(actual_min.date()),
    "actual_stock_max_date": str(actual_max.date()),
    "actual_nifty_min_date": str(nifty["date"].min().date()),
    "actual_nifty_max_date": str(nifty["date"].max().date()),
    "expected_min_end_date": str(EXPECTED_MIN_END_DATE.date()),
    "starting_capital": STARTING_CAPITAL,
    "max_positions": MAX_POSITIONS,
    "entry_score_min": ENTRY_SCORE_MIN,
    "trailing_stops": TRAILING_STOPS,
    "hard_initial_stop": False,
    "failure_exit_days": FAILURE_EXIT_DAYS,
    "max_hold_days": MAX_HOLD_DAYS,
    "slippage_bps": SLIPPAGE_BPS,
    "buy_cost_bps": BUY_COST_BPS,
    "sell_cost_bps": SELL_COST_BPS,
    "nifty_regime_definition": {
        "bull": [
            "Nifty close > 200DMA",
            "Nifty 50DMA > 200DMA",
            "Nifty 20D return > 0"
        ],
        "bear": [
            "Nifty close < 200DMA",
            "Nifty 50DMA < 200DMA",
            "Nifty 20D return < 0"
        ],
        "mixed": "Everything else"
    },
    "execution_model": "close_T_decision_to_open_T_plus_1_execution",
    "walk_forward_train_years": WF_TRAIN_YEARS,
    "walk_forward_validation_years": WF_VALID_YEARS,
    "walk_forward_test_years": WF_TEST_YEARS,
}

MANIFEST_PATH = RESULTS_DIR / "v4_run_manifest.json"

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(json.dumps(manifest, indent=2, default=str))


In [ ]:
print("=" * 80)
print("NSE 5× TURNAROUND V4 — FINAL SUMMARY")
print("=" * 80)

print("Stock dataset:", actual_min.date(), "→", actual_max.date())
print("Nifty dataset:", nifty["date"].min().date(), "→", nifty["date"].max().date())
print("Stock symbols:", f"{stocks['symbol'].nunique():,}")
print("Raw entry signals:", f"{int(features['entry_signal'].sum()):,}")

print("\nNifty regime distribution:")
display(regime_distribution)

print("\nFull strategy matrix:")
display(
    matrix[
        [
            "regime", "trailing_stop", "ending_equity",
            "CAGR", "max_drawdown", "Sharpe",
            "closed_trades", "avg_exposure"
        ]
    ].sort_values(["regime", "trailing_stop"])
)

print("\nResults directory:")
print(RESULTS_DIR)
print("\n✓ V4 backtest completed.")
